# Classify with tabular features

**What it does.** Fit a gradient-boosted / random-forest model on the measured features rather than on the images.

**When to use it.** When the phenotype is captured by measurements you already have. Far cheaper than a network, and the feature importances are readable.

**What you get.** Per-object scores, a feature-importance table and a SHAP summary.

---

> Every path below is a placeholder. Point `src` at your own data before running.
> Nothing in this notebook writes outside the folder you give it.

## 1. Check the install

If this cell fails, the rest cannot work. It reports the version and whether a GPU is visible — segmentation and training are usable on CPU but slow.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. The function this notebook runs

`spacr.ml.ml_analysis`

```
ml_analysis(df, channel_of_interest=3, location_column='columnID', positive_control='c2', negative_control='c1', exclude=None, n_repeats=10, top_features=30, reg_alpha=0.1, reg_lambda=1.0, learning_rate=1e-05, n_estimators=1000, test_size=0.2, model_type='xgboost', n_jobs=-1, remove_low_variance_features=True, remove_highly_correlated_features=True, prune_features=False, cross_validation=False, verbose=False, *, batch_correction='none', batch_column='plateID', batch_control_column=None, batch_control_values=None, batch_covariate_column=None, batch_combat_mean_only=False, batch_min_samples=3, batch_missing_control='error')
```

Train a per-object classifier on positive/negative control wells and score every row of the input DataFrame.

In [ ]:
from spacr.ml import ml_analysis

## 3. Settings

`spacr.settings.set_default_analyze_screen` fills in every default, so you only have to write down what differs. The cell below prints the full set as it exists in this version — treat that output as the reference, not this notebook.

Change values in `settings`, not in the defaults helper.

In [ ]:
from spacr.settings import set_default_analyze_screen

defaults = set_default_analyze_screen({})
for key in sorted(defaults):
    print(f'{key:38s} {defaults[key]!r}')

### Every setting this function accepts

The full dictionary, each key on its own line with its default and what it controls. Edit values in place; delete nothing — a key left at its default behaves exactly as if it were absent.

Generated from this installed version, so it is the real set of keys, the real defaults and the real descriptions.

In [ ]:
settings = {
    # (str) - Name of the integer column in the png_list table that
    # holds manual class calls; the Annotate app adds it with ALTER
    # TABLE if it is missing and writes labels into it. It is the ground
    # truth when dataset_mode is 'annotation' (used as the fallback when
    # annotation_columns is unset), and in ML screen analysis it
    # replaces location_column so controls are taken from annotations
    # rather than plate position. Default 'test' in the
    # dataset-generation and Annotate settings, but None in Analyze
    # Screen - and that None is exactly what leaves the location_column
    # override switched off, so the screen-analysis behaviour is opt-in.
    'annotation_column': None,

    # (str) - Metadata column that identifies independent acquisition
    # batches, normally 'plateID'. Every analyzed row must have a value
    # and at least batch_min_samples rows must occur in each batch. Use
    # an acquisition date or instrument ID only if that is the nuisance
    # source you intend to remove. Default 'plateID'. API:
    # spacr.batch_correction.correct_batch_effects.
    'batch_column': 'plateID',

    # (bool) - True corrects only the additive batch shift and leaves
    # each batch's scale alone. Use it when the plates differ in level
    # but not in spread, or when a batch has too few rows for a stable
    # variance estimate. False (the default) corrects both location and
    # scale, which is standard ComBat. Ignored by every method other
    # than combat. API: spacr.batch_correction.correct_batch_effects.
    'batch_combat_mean_only': False,

    # (str or None) - Metadata column containing reference-control
    # labels for control_center, normally 'columnID' for plate controls.
    # It is ignored by center, zscore, robust_zscore, and none. Blank
    # follows col_to_compare in Image UMAP or location_column in
    # Classify (ML); regression defaults to 'columnID'. API:
    # spacr.batch_correction.correct_batch_effects.
    'batch_control_column': None,

    # (str, number, list or None) - Reference/negative-control value(s)
    # in batch_control_column used by control_center. Each plate needs
    # at least batch_min_samples matching rows. Image UMAP falls back to
    # neg and Classify (ML) to negative_control when this field is
    # blank; regression requires an explicit value. Default varies by
    # module. API: spacr.batch_correction.correct_batch_effects.
    'batch_control_values': None,

    # (str) - Optional plate/batch correction applied before Image UMAP,
    # ML screen classification, or phenotype regression. 'none' leaves
    # measurements unchanged; 'center' removes each plate's mean shift;
    # 'zscore' aligns plate means and variances; 'robust_zscore' uses
    # median/MAD and tolerates outliers; 'control_center' estimates only
    # a location shift from reference controls and best preserves
    # treatment dispersion; 'combat' is empirical-Bayes ComBat and needs
    # batch_covariate_column naming the biology to protect, or it
    # refuses to run. Do not correct when plate is confounded with
    # biology. Default 'none'. API:
    # spacr.batch_correction.correct_batch_effects.
    'batch_correction': 'none',

    # (str, list or None) - Metadata column(s) naming the biology that
    # combat must protect, e.g. 'condition' or 'condition,timepoint'.
    # combat estimates the batch effect from the residuals after these
    # terms, so anything NOT listed here is treated as noise and removed
    # with the plate effect -- leave your treatment out and combat
    # deletes the contrast you are measuring. Required by combat and
    # ignored by every other method; blank makes combat refuse to run
    # rather than silently destroy the signal. Write 'none' to state
    # deliberately that there is no covariate to protect. Default None.
    # API: spacr.batch_correction.correct_batch_effects.
    'batch_covariate_column': None,

    # (int) - Minimum number of rows required in every batch, and
    # minimum matching reference controls per batch for control_center.
    # Correction stops with an actionable error below this threshold
    # because a one- or two-object plate estimate is unstable. Default
    # 3. API: spacr.batch_correction.correct_batch_effects.
    'batch_min_samples': 3,

    # (str) - Policy when control_center cannot find enough reference
    # controls on a plate: 'error' stops rather than silently mixing
    # corrected and raw plates; 'skip' leaves that plate unchanged and
    # records a warning. Default 'error'. API:
    # spacr.batch_correction.correct_batch_effects.
    'batch_missing_control': 'error',

    # (int) - Index of the fluorescence channel the downstream analysis
    # focuses on. It decides which channel's features survive filtering
    # (other channels' features are dropped), defines recruitment =
    # pathogen_channel_N_mean_intensity /
    # cytoplasm_channel_N_mean_intensity, and is written into the ML
    # result paths. Set it to the channel carrying your phenotype
    # readout. Valid 0-3; default 3 in the ML/recruitment steps, 1-2
    # elsewhere.
    'channel_of_interest': 3,

    # (str) - Matplotlib colormap applied to single-channel image
    # previews and to plate heatmaps. Perceptually uniform maps
    # ('viridis', 'inferno', 'magma') keep intensity differences honest;
    # 'gray' matches how the raw microscope data looks. Any registered
    # matplotlib name works, with an '_r' suffix to reverse it. Default
    # 'inferno' for image plots, 'viridis' for plate heatmaps.
    'cmap': 'viridis',

    # (bool) - Score the classifier with 5-fold stratified
    # cross-validation instead of a single train/test split, so every
    # control object receives an out-of-fold prediction and an optimal
    # probability threshold is picked per fold. Gives a far more stable
    # accuracy estimate on small control sets, at roughly 5x the
    # training time. Default True.
    'cross_validation': True,

    # (str or list) - Names of measurement columns to drop from the
    # feature set before UMAP embedding or ML training, applied after
    # the channel_of_interest selection. Use it to remove features that
    # leak the label or swamp the embedding. It does not filter database
    # rows; use exclude_rows for that. Default None keeps every feature.
    'exclude': None,

    # (str) - How per-object values collapse to one number per well in
    # the plate heatmap: 'mean' averages heatmap_feature over the
    # objects in a well, 'sum' totals them, 'count' ignores the feature
    # and colors wells by object count. Use 'count' to spot uneven
    # seeding or dropout, 'mean' for phenotype strength. Default 'mean';
    # any other value raises ValueError.
    'grouping': 'mean',

    # (str) - Numeric column that is aggregated per well and
    # color-mapped in the plate heatmap after ML scoring, e.g.
    # 'predictions' for the classifier score or 'recruitment' for the
    # pathogen/cytoplasm intensity ratio. Must be a numeric column of
    # the scored dataframe or the run raises ValueError listing the
    # valid names. Default 'predictions'.
    'heatmap_feature': 'predictions',

    # (float) - Step size passed to the optimizer. Too high and the loss
    # spikes or flatlines at chance; too low and training crawls or
    # settles in a poor minimum. 1e-3 suits training from scratch, while
    # 1e-4 to 1e-5 is safer when fine-tuning ImageNet weights
    # (init_weights=True). The chosen schedule decays this starting
    # value over the run. Default 0.001.
    'learning_rate': 0.001,

    # (str) - Metadata column searched for the positive_control and
    # negative_control values when labelling rows for ML training,
    # normally 'columnID' or 'rowID'. Set 'rowID' when your controls run
    # along plate rows instead of columns. It is overwritten with
    # annotation_column whenever that is set. Default 'columnID'.
    'location_column': 'columnID',

    # (str) - Color limits for the plate heatmap: 'allq' scales to the
    # 2nd-98th percentile of well values so a handful of extreme wells
    # cannot flatten the rest, 'all' scales to the true min and max. A
    # two-element list is also accepted, where floats are read as
    # quantiles and integers as absolute vmin/vmax. Default 'allq'.
    'min_max': 'allq',

    # (int) - Wells with fewer than this many measured cells are removed
    # before the ML plate heatmap is built. They are not left blank: the
    # pivot is filled with 0 afterwards, so excluded wells render at the
    # bottom of the colour scale and look like a genuine zero, and the
    # 'allq' 2-98 percent limits are taken over that zero-filled matrix.
    # It affects this heatmap only - the classifier and the saved
    # results table still use every well. Set 0 to switch the filter
    # off. Default 25.
    'minimum_cell_count': 25,

    # (str) - Which classifier ml_analysis fits to separate positive-
    # from negative-control wells and rank per-object features by
    # permutation importance. One of xgboost (default), lightgbm,
    # catboost, random_forest, extra_trees, gradient_boosting,
    # logistic_regression, svm, mlp; lightgbm and catboost need their
    # optional packages. reg_alpha, reg_lambda and learning_rate only
    # affect the boosted models; logistic_regression is a good linear
    # sanity check.
    'model_type_ml': 'xgboost',

    # (int) - Number of trees or boosting rounds in the tabular ML
    # classifier - n_estimators for
    # RandomForest/ExtraTrees/XGBoost/LightGBM, iterations for CatBoost,
    # max_iter for HistGradientBoosting. More rounds keep improving fit
    # up to a plateau while training time grows linearly; boosted models
    # can overfit past it. Default 1000.
    'n_estimators': 1000,

    # (int) - CPU workers for parallel stages: measurement, mask
    # adjustment, DataLoader loading, and the sklearn/UMAP calls where
    # -1 means every core. Raise it to shorten CPU-bound steps until RAM
    # or disk I/O saturates. Note the measure-and-crop pipeline
    # overrides your value with cpu_count()-4. Defaults vary by
    # pipeline: cpu_count()-4, -1, or None.
    'n_jobs': -1,

    # (int) - Number of times each feature is randomly shuffled when
    # computing permutation importance for the ML classifier. More
    # repeats shrink the error bars on the importance ranking but cost
    # an extra full prediction pass per feature per repeat. Default 10;
    # drop to 3-5 for a quick look at wide feature tables.
    'n_repeats': 10,

    # (str) - Identifier of the negative-control class. In ML screening
    # it is the value in location_column (e.g. 'c1') whose objects are
    # labelled class 0 for training; in gRNA regression it is a
    # gene/gRNA ID substring (e.g. '233460') matched against coefficient
    # names to tag them 'nc' in the results and volcano plot. Defaults
    # 'c1' and '233460' respectively.
    'negative_control': 'c1',

    # (int, bool, or None) - Cap on nuclei per cell applied when the
    # per-object tables are merged for analysis: None disables the
    # filter, True keeps only single-nucleus cells, an integer N keeps
    # cells with N or fewer nuclei. Cells over the cap are dropped
    # entirely from the merged table. Do not pass False - it is read as
    # 0 and removes everything. Defaults differ sharply by pipeline: 1
    # for plot-merge and recruitment, 2 for plot-data-from-db, True for
    # screen analysis and training-dataset generation, 10 for
    # endodyogeny, and 1000 (effectively off) for vision-model
    # interpretation and class-proportion analysis - check the pipeline
    # you are running rather than assuming.
    'nuclei_limit': True,

    # (int, bool, or None) - Maximum pathogens per cell. True or 1 =
    # single pathogen only; None or False = no limit; int = custom
    # limit.
    'pathogen_limit': 3,

    # (str) - Identifier of the positive-control class. In ML screening
    # it is the value in location_column (e.g. 'c2') whose objects are
    # labelled class 1 for training; in gRNA regression it is a
    # gene/gRNA ID substring (e.g. '239740') matched against coefficient
    # names to tag them 'pc' in the results and volcano plot. Defaults
    # 'c2' and '239740' respectively.
    'positive_control': 'c2',

    # (bool) - Before training, keep only the top_features columns with
    # the highest ANOVA F-score against the control labels (sklearn
    # SelectKBest with f_classif). Speeds up fitting and can curb
    # overfitting on small control sets, but discards features the model
    # might have used and scores each feature in isolation, ignoring
    # interactions. Default False.
    'prune_features': False,

    # (float) - L1 penalty on leaf weights for the gradient-boosted
    # classifier (XGBoost and LightGBM; ignored by the other
    # model_type_ml choices). Raising it drives more leaf weights to
    # exactly zero, shrinking the model and its effective feature set -
    # raise it when training accuracy far exceeds test accuracy. Any
    # value >= 0. Default 0.1.
    'reg_alpha': 0.1,

    # (float) - L2 penalty on leaf weights for the gradient-boosted
    # classifier (XGBoost, LightGBM, and CatBoost's l2_leaf_reg).
    # Raising it shrinks all weights smoothly rather than zeroing them,
    # damping the influence of any single feature and curbing
    # overfitting, at the risk of underfitting if pushed too far. Any
    # value >= 0. Default 1.0.
    'reg_lambda': 1.0,

    # (bool) - In the machine-learning feature table, drop any feature
    # whose absolute Pearson correlation with an already-kept feature
    # exceeds 0.95, applied after the channel_of_interest filter. Leave
    # it on so redundant measurements do not split importance scores and
    # slow fitting; turn it off only when you need every original
    # column. Default True. Note the UMAP path uses
    # remove_highly_correlated instead.
    'remove_highly_correlated_features': True,

    # (bool) - Drop numeric features whose variance across objects falls
    # below 0.01 before model fitting -- near-constant columns that
    # carry no discriminative signal but still cost time and dilute
    # importance rankings. Turn it off only when your features live on a
    # very small numeric scale, where genuine signal can fall under that
    # fixed cut-off. Default True.
    'remove_low_variance_features': True,

    # (bool) - After ML screen analysis, write the per-object model
    # scores back into measurements.db as a 'predictions' column on the
    # png_list table, matched on prcfo. Enable when you want to sort,
    # filter or plot objects by score in the GUI; the CSV result files
    # are written either way. Default False.
    'save_to_db': False,

    # (str, path) - Folder the current step reads from and writes into:
    # raw images for mask generation, the merged/ folder of .npy stacks
    # for measure, the plate root for dataset/regression steps, or the
    # folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/,
    # measurements/measurements.db, datasets/, results/) are created
    # inside it. A list of paths, or a "['a','b']" string, processes
    # several plates in one run.
    'src': 'path',

    # (float) - Fraction of the labelled single-object rows held out as
    # the test split in the tabular ML classifier; the remainder trains
    # the model. Raise it for a more trustworthy accuracy estimate,
    # lower it when labelled data is scarce and you need the rows for
    # training. Valid 0-1, default 0.2 (20% test).
    'test_size': 0.2,

    # (int) - Feature cap in the ML screen analysis: how many rows the
    # feature-importance and permutation-importance bar plots show, and
    # how many top-ranked features the SHAP refit and its summary plot
    # use. It is also the k of the SelectKBest pruning applied before
    # the model is fitted, but only when prune_features is True - with
    # prune_features at its default False the classifier trains on every
    # feature and this is reporting/SHAP scope only. Raise for a fuller
    # picture, lower for readable plots. Default 30.
    'top_features': 30,

    # (bool) - Print extra run detail instead of the minimal log: the
    # resolved settings table at the start of mask generation, the
    # channel and Cellpose-model choices per object type, per-table row
    # counts and how many objects survive the nuclei/pathogen-per-cell
    # filters when measurement tables are merged, and extra
    # loader/diagnostic output in the training and UMAP paths. It only
    # adds console output, so turn it on when object counts come out
    # unexpected and you need to see which stage removed them. Defaults
    # are per-pipeline: True for mask generation, UMAP, screen analysis,
    # barcode mapping, Cellpose training and plaque analysis; False for
    # measure-and-crop, plot-from-db and plot-from-CSV, the endodyogeny
    # and class-proportion helpers, the Cellpose check/finetune tools,
    # and the screen regression, whose verbose branch display()s the
    # whole per-object score table.
    'verbose': True,

}

# Fill in anything left unset, then check the source path.
settings = set_default_analyze_screen(settings)
settings['src']

## 4. Run it

This is the long cell. Progress is logged; if you want more of it, raise the log levels in Preferences → Logging, or set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter.

In [ ]:
ml_analysis(settings)

## Where the output went

Per-object scores, a feature-importance table and a SHAP summary.

spaCR writes beside the source folder rather than into a global location, so a plate stays self-contained and re-running does not clobber a different experiment.

### Next steps

* The GUI covers the same workflows with the settings laid out as a form — `python -m spacr`.
* The narrated walkthroughs are at <https://einarolafsson.github.io/spacr/tutorials/>.
* The API reference is at <https://einarolafsson.github.io/spacr/>.